# Imports

In [1]:
from langchain_community.llms import LlamaCpp
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA

# Config

In [ ]:
GENERATION_MODEL_PATH = 'Phi-3-mini-4k-instruct-fp16.gguf'
RETREIVAL_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

In [3]:
FACTUAL_DATA = """
A science fiction epic released in 2016, titled 'Cosmos Voyage', was 
co-written and directed by Ethan Caldwell. The cast includes Liam 
McFarland, Emma Hartley, Julia Stanton, Ben Irwin, Ellen Barrett, Matt 
Lawson, and Michael Kane. The story is set in a bleak future where humanity 
is facing extinction. A team of astronauts embarks on a journey through a 
mysterious wormhole located near Jupiter, seeking a new planet capable of 
sustaining human life. The screenplay was created by Ethan and his brother 
Jonathan Caldwell, building on a story Jonathan had drafted in 2008. The 
project had scientific guidance from Dr. Alan Pierce, a theoretical 
physicist at MIT and 2018 Nobel laureate in Physics. He acted as an executive 
producer, offered scientific consultancy, and published a companion book, 
'The Science Behind Cosmos Voyage'. Cinematography was led by Hoyte Van 
Lytton, who shot on 35 mm Panavision film and 70 mm IMAX. Principal 
photography started in early 2015, with locations including Alberta, 
Greenland, and Los Angeles. The film used a combination of practical effects, 
miniatures, and digital visual effects created by the studio Digital Mirage. 
'Cosmos Voyage' premiered on November 12, 2016, initially on traditional 
film reels, before expanding to theaters with digital projection. It generated income 
over $712 million worldwide ($790 million including subsequent re-releases), 
making it one of the top-grossing films of 2016. Critics praised its 
performances, direction, screenplay, musical score, visual effects, thematic 
ambition, and emotional impact. Many astronomers also lauded its accurate 
depiction of astrophysical concepts. Since release, the movie has developed 
a dedicated fan following and is widely regarded as a standout in modern 
science fiction cinema. At the 89th Academy Awards, it received five 
nominations, winning Best Visual Effects, along with several other accolades."""

In [5]:
len(FACTUAL_DATA)

1847

# Download Model

In [6]:
!wget --no-clobber "https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf"

File ‘Phi-3-mini-4k-instruct-fp16.gguf’ already there; not retrieving.



# RAG System

## Generation Model

In [37]:
llm = LlamaCpp(
    model_path=GENERATION_MODEL_PATH,
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False,
)

## Retrieval Model

In [9]:
texts = FACTUAL_DATA.split('.')
texts = [t.strip(' \n') for t in texts]

In [7]:
embedding_model = HuggingFaceEmbeddings(model_name=RETREIVAL_MODEL_NAME)

/tmp/ipykernel_869549/1561884496.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding_model = HuggingFaceEmbeddings(model_name=RETREIVAL_MODEL_NAME)
/home/mmahdi/.cache/pypoetry/virtualenvs/rag-example-FwGg7RSF-py3.10/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
No sentence-transformers model found with name bigscience/bloomz-560m. Creating a new one with mean pooling.


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

In [10]:
retreival_db = FAISS.from_texts(texts, embedding_model)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [11]:
retriver = retreival_db.as_retriever()

retriver.get_relevant_documents("فیلم سفر کیهانی چقدر درآمد ایجاد کرد؟")

/tmp/ipykernel_869549/1360411665.py:3: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  retriver.get_relevant_documents("فیلم سفر کیهانی چقدر درآمد ایجاد کرد؟")


[Document(page_content='بسیاری از اخترشناسان نیز نمایش دقیق مفاهیم اخترفیزیکی را تحسین کردند'),
 Document(page_content='او به عنوان تهیه\u200cکننده اجرایی فعالیت کرد، مشاوره علمی ارائه داد و کتاب همراهی با عنوان «علم پشت سفر کیهانی» منتشر کرد'),
 Document(page_content='فیلم\u200cبرداری اصلی اوایل سال ۲۰۱۵ آغاز شد و لوکیشن\u200cها شامل اردبیل، قشم و تهران بودند'),
 Document(page_content='از زمان انتشار، فیلم طرفداران وفاداری پیدا کرده و به عنوان یکی از آثار برجسته در سینمای علمی-تخیلی مدرن شناخته می\u200cشود')]

## RAG Prompt

In [64]:
template = """<|user|>
Relevant information:
{context}
Provide a concise answer the following question using the relevant information
provided above:
{question}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"],
)

## RAG Pipeline

In [65]:
rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retreival_db.as_retriever(),
    chain_type_kwargs={
        "prompt": prompt
    },
    verbose=True,
)

## Invoke

In [55]:
rag.invoke('What is the income generated by Cosmos Voyage film?')



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'What is the income generated by Cosmos Voyage film?',
 'result': " The 'Cosmos Voyage' film generated an income of over $712 million worldwide, with $790 million including subsequent re-releases."}

In [30]:
rag.invoke('What is the income generated by God Father?')



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'What is the income generated by God Father?',
 'result': ' I\'m sorry for the confusion, but there seems to be an error. "Godfather" refers to The Godfather film series, particularly The Godfather (1972) and its sequel, The Godfather Part II (1974). Based on your provided details, it appears you might be referring to a different project with similar elements or perhaps another movie that earned over $712 million worldwide in 2016. However, using the information given, we can\'t accurately determine an income for "The Godfather" as the specified figures are related to later re-releases and do not pertain directly to the original films\' earnings at their time of release.\n\nIf you meant another movie with a similar name or involved Dr. Jonathan Caldwell, please provide additional details so I can assist you further. For accurate financial data on "The Godfather" (1972) and its sequel, one could refer to historical box office records which show that The Godfather earned over $

In [56]:
rag.invoke('Which awards has Cosmos Voyage won؟')



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Which awards has Cosmos Voyage won؟',
 'result': ' Cosmos Voyage won the Academy Award for Best Visual Effects at the 89th Academy Awards. It also received several other accolades, although specific additional awards are not listed in the provided information.'}

In [58]:
rag.invoke('Who are the actors in Cosmos Voyage?')



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Who are the actors in Cosmos Voyage?',
 'result': ' The actors in "Cosmos Voyage" are Liam McFarland, Emma Hartley, Julia Stanton, Ben Irwin, Ellen Barrett, Matt Lawson, and Michael Kane.'}